# PhDAI 732 Group Project Part 1
**Group 5** | PhiUSIIL Phishing URL Dataset

This notebook is a thin driver. All logic lives in `src/`. If you find yourself
writing a function in a cell, it belongs in a module instead, or the next person
to run this notebook gets different numbers than you did.

In [ ]:
# Setup. Run once per Colab session.
REPO = 'https://github.com/jecollier041/phdai_732_gp_phiusil.git'
DIR  = 'phdai_732_gp_phiusil'

import os, sys
if not os.path.exists(DIR):
    !git clone -q $REPO
%cd $DIR
!pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())

# Sanity check: this must print the repo root, and src must import.
from src import config as C
print('ROOT      :', C.ROOT)
print('RAW_CSV   :', C.RAW_CSV, '| present:', C.RAW_CSV.exists())
print('SEED      :', C.SEED, '| TEST_SIZE:', C.TEST_SIZE)


In [ ]:
# Fetch the dataset into data/. Do not commit the CSV.
!pip install -q ucimlrepo
import pandas as pd
from src import config as C

if not C.RAW_CSV.exists():
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=967)
    frame = ds.data.original.copy()
    frame = frame.loc[:, ~frame.columns.duplicated()]
    C.RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(C.RAW_CSV, index=False)
    # ucimlrepo splits identifier columns off into ds.data.ids on some
    # datasets. If FILENAME/URL/Domain/Title are missing, load_data will raise
    # a KeyError naming them, which is the signal that this cell needs fixing
    # rather than config.py.
print(C.RAW_CSV, C.RAW_CSV.exists())


In [ ]:
# Step 1 (Data steward): load, clean, document.
from src.data import load_data

df, log = load_data()
log

In [ ]:
# Step 2 (Data steward): leakage screen. Run before anyone models.
from src.leakage import single_feature_accuracy, class_constancy

screen = single_feature_accuracy(df)
display(screen.head(12))
class_constancy(df).head(8)

In [ ]:
# Step 3: freeze the split. Do this once. Commit results/split_assignment.csv.
# Each strategy caches to its own file, so running the grouped split below
# does not silently hand back the stratified one.
from src.splits import make_split, get_xy

assignment = make_split(df, strategy='stratified')
print(assignment.value_counts().to_dict())

# Domain-grouped comparison. Separate cache, separate numbers.
assignment_grouped = make_split(df, strategy='grouped')
print(assignment_grouped.value_counts().to_dict())


In [ ]:
# Step 4 (Modeling lead): all models across all three feature sets.
# Runtime note: random_forest is ~100 s per feature set on a fast multicore
# box and several times that on a free Colab CPU runtime. Budget ~45 min for
# this cell, or pass cv=False to evaluate() while iterating.
from src.models import build_models, evaluate
import pandas as pd

records = []
for fs in ['full', 'no_derived', 'url_only']:
    Xtr, Xte, ytr, yte = get_xy(df, assignment, feature_set=fs)
    for name, model in build_models().items():
        records.append(evaluate(model, name, fs, Xtr, Xte, ytr, yte))

results = pd.DataFrame(records)
results[['feature_set','model','accuracy','precision','recall','f1','roc_auc']]


## Handoff

Everything the report needs is now on disk:

- `results/cleaning_log.json` feeds the preprocessing half of Section 2
- `results/leakage_screen.csv` and `class_constancy.csv` feed the analytical argument
- `results/metrics_*.json` feed Section 3
- `figures/*.png` feed both

Report writers read these files. Do not retype numbers.